# Description

In this notebook, I will explore the different between different hierarchical quantization, including:
- Vector quantization.
- Matrix quantization.
- Tensor quantization.

In [1]:
import os 
import torch 
import bitsandbytes as bnb

In [2]:
def quantization_error_l2_norm(original, dequantized):
    """
    Compute the relative error between the original and dequantized tensors using l2 norm.
    """
    return torch.norm(original - dequantized)


def quantization_error_mse(original, dequantized):
    """
    Compute the Mean Squared Error (MSE) between the original and dequantized tensors.
    """
    return torch.mean((original - dequantized) ** 2)


def quantization_error_kl_divergence(original, dequantized, num_bins=2048, epsilon=1e-10):
    """
    Compute the KL divergence between the distributions of the original and dequantized tensors with float16.
    """
    orig_hist = torch.histc(original.float(), bins=num_bins, min=-1.0, max=1.0)
    deq_hist = torch.histc(dequantized.float(), bins=num_bins, min=-1.0, max=1.0)

    orig_prob = orig_hist / (torch.sum(orig_hist) + epsilon)
    deq_prob = deq_hist / (torch.sum(deq_hist) + epsilon)

    kl_div = torch.sum(orig_prob * torch.log((orig_prob + epsilon) / (deq_prob + epsilon)))
    return kl_div

# 1. Vector quantization

In [3]:
def quantize_row_matrix_int8_symmetric(mat:torch.Tensor):
    """
    Symmetric quantization to int8 on a per-row basis.
    mat: input float tensor (e.g., torch.float32 or torch.bfloat16)
    """
    N, M = mat.shape
    qmin = -128
    qmax = 127
    
    max_vals, _ = torch.max(torch.abs(mat), dim=1, keepdim=True)  # shape (N, 1)
    scales = (max_vals / qmax).squeeze(1)  # shape (N,)
    
    q_mat = torch.clamp(torch.round(mat / scales.unsqueeze(1)), qmin, qmax).to(torch.int8)  # shape (N, M)
    
    scales = scales.to(torch.float32)
    return q_mat, scales

def de_quantize_row_matrix_int8_symmetric(q_mat:torch.Tensor, scale:torch.Tensor, out_dtype=torch.float16):
    """
    Dequantize int8 matrix to float on a per-row basis.
    q_mat: input int8 tensor (shape (N, M))
    scales: scale factors for each row (shape (N,))
    """
    output = q_mat.to(torch.float32) 
    output = output * scale[:, None]
    output = output.to(out_dtype)
    return output

In [4]:
d_type = torch.float16
N = 512

W = torch.randn(N, N, device='cuda', dtype=d_type)

In [5]:
W_q, w_scale = quantize_row_matrix_int8_symmetric(W)

print(f"W_q shape: {W_q.shape}, dtype: {W_q.dtype}")
print(f"w_scale shape: {w_scale.shape}, dtype: {w_scale.dtype}")

W_q shape: torch.Size([512, 512]), dtype: torch.int8
w_scale shape: torch.Size([512]), dtype: torch.float32


In [6]:
W_deq = de_quantize_row_matrix_int8_symmetric(W_q, w_scale)
print(f"Shape of A_deq: {W_deq.shape}, dtype: {W_deq.dtype}")

Shape of A_deq: torch.Size([512, 512]), dtype: torch.float16


In [7]:
if torch.allclose(W, W_deq, rtol=0.1, atol=0.1):
    print("Correct !!")
else:
    print("WRONG - Dequantized matrix is NOT close !!")
    
error_l2 = quantization_error_l2_norm(W, W_deq)
print(f"Quantization L2 norm error (row-wise symmetric int8): {error_l2.item():.6f}")
    
error_mse = quantization_error_mse(W, W_deq)
print(f"Quantization MSE (row-wise symmetric int8): {error_mse.item():.6f}")

error_kl = quantization_error_kl_divergence(W, W_deq)
print(f"Quantization KL divergence (row-wise symmetric int8): {error_kl.item():.6f}")

Correct !!
Quantization L2 norm error (row-wise symmetric int8): 3.804688
Quantization MSE (row-wise symmetric int8): 0.000055
Quantization KL divergence (row-wise symmetric int8): 0.428449


# 2. Matrix quantization

In [8]:
def quantize_matrix_symmetric_int8(mat:torch.Tensor):
    """
    Symmetric quantization to int8.
    mat: input float matrix (e.g., torch.float32 or torch.bfloat16)
    """
    max_val = torch.max(torch.abs(mat))
    
    qmin = -128
    qmax = 127
    scale = max_val / qmax
    
    q_mat = torch.clamp(torch.round(mat / scale), qmin, qmax).to(torch.int8)
    
    scale = torch.tensor(scale, dtype=torch.float32)
    return q_mat, scale

def de_quantize_matrix_symmetric_int8(q_mat:torch.Tensor, scale:torch.Tensor, out_dtype=torch.float16):
    """
    Dequantize int8 tensor to float.
    q_mat: input int8 tensor
    scale: scale factor (single value)
    """
    output = q_mat.to(torch.float32) 
    output = output * scale
    output = output.to(out_dtype)
    return output

In [9]:
d_type = torch.float16
N = 1024

W = torch.randn(N, N, device='cuda', dtype=d_type)
print(f"W shape: {W.shape}, dtype: {W.dtype}")

W shape: torch.Size([1024, 1024]), dtype: torch.float16


In [10]:
q_W, scales = quantize_matrix_symmetric_int8(W)
print(f"Shape of q_W: {q_W.shape}, Shape of scales: {scales.shape}")

Shape of q_W: torch.Size([1024, 1024]), Shape of scales: torch.Size([])


/tmp/ipykernel_326260/2675896001.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  scale = torch.tensor(scale, dtype=torch.float32)


In [11]:
W_deq = de_quantize_matrix_symmetric_int8(q_W, scales)
print(f"Shape of W_deq: {W_deq.shape}, dtype: {W_deq.dtype}")

Shape of W_deq: torch.Size([1024, 1024]), dtype: torch.float16


In [12]:
if torch.allclose(W, W_deq, rtol=0.1, atol=0.1):
    print("Correct !! \n")
else:
    print("WRONG - Dequantized matrix is NOT close !! \n")
    
error_l2 = quantization_error_l2_norm(W, W_deq)
print(f"Quantization L2 norm error (symmetric int8): {error_l2.item():.6f}")

error_mse = quantization_error_mse(W, W_deq)
print(f"Quantization MSE (symmetric int8): {error_mse.item():.6f}")

error_kl = quantization_error_kl_divergence(W, W_deq)
print(f"Quantization KL divergence (symmetric int8): {error_kl.item():.6f}")

Correct !! 

Quantization L2 norm error (symmetric int8): 12.734375
Quantization MSE (symmetric int8): 0.000155
Quantization KL divergence (symmetric int8): 14.971045


# 3. Tensor quantization

In [13]:
def quantize_tensor_symmetric_int8(mat:torch.Tensor):
    """
    Symmetric quantization to int8.
    mat: input float matrix (e.g., torch.float32 or torch.bfloat16)
    """
    max_val = torch.max(torch.abs(mat))
    
    qmin = -128
    qmax = 127
    scale = max_val / qmax
    
    q_mat = torch.clamp(torch.round(mat / scale), qmin, qmax).to(torch.int8)
    
    scale = torch.tensor(scale, dtype=torch.float32)
    return q_mat, scale

def de_quantize_tensor_symmetric_int8(q_mat:torch.Tensor, scale:torch.Tensor, out_dtype=torch.float16):
    """
    Dequantize int8 tensor to float.
    q_mat: input int8 tensor
    scale: scale factor (single value)
    """
    output = q_mat.to(torch.float32) 
    output = output * scale
    output = output.to(out_dtype)
    return output

In [14]:
H = 16
N = 512
d_type = torch.float16

W = torch.randn((H, N, N), device='cuda', dtype=d_type)
print(f"W shape: {W.shape}, dtype: {W.dtype}")

W shape: torch.Size([16, 512, 512]), dtype: torch.float16


In [15]:
W_q, scales = quantize_tensor_symmetric_int8(W)
print(f"Shape of W_q: {W_q.shape}, Shape of scales: {scales.shape}")

Shape of W_q: torch.Size([16, 512, 512]), Shape of scales: torch.Size([])


/tmp/ipykernel_326260/315809518.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  scale = torch.tensor(scale, dtype=torch.float32)


In [16]:
W_deq = de_quantize_tensor_symmetric_int8(W_q, scales)
print(f"Shape of W_deq: {W_deq.shape}, dtype: {W_deq.dtype}")

Shape of W_deq: torch.Size([16, 512, 512]), dtype: torch.float16


In [17]:
if torch.allclose(W, W_deq, rtol=0.1, atol=0.1):
    print("Correct !! \n")
else:
    print("WRONG - Dequantized matrix is NOT close !! \n")
    
error_l2 = quantization_error_l2_norm(W, W_deq)
print(f"Quantization L2 norm error (symmetric int8): {error_l2.item():.6f}")

error_mse = quantization_error_mse(W, W_deq)
print(f"Quantization MSE (symmetric int8): {error_mse.item():.6f}")

error_kl = quantization_error_kl_divergence(W, W_deq)
print(f"Quantization KL divergence (symmetric int8): {error_kl.item():.6f}")

Correct !! 

Quantization L2 norm error (symmetric int8): 26.328125
Quantization MSE (symmetric int8): 0.000165
Quantization KL divergence (symmetric int8): 14.989372


# 4. Matmul - Vector quantization

In [ ]:
def quantize_row_matrix_int8_symmetric(mat:torch.Tensor):
    """
    Symmetric quantization to int8 on a per-row basis.
    mat: input float tensor (e.g., torch.float32 or torch.bfloat16)
    """
    N, M = mat.shape
    qmin = -128
    qmax = 127
    
    max_vals, _ = torch.max(torch.abs(mat), dim=1, keepdim=True)  # shape (N, 1)
    scales = (max_vals / qmax).squeeze(1)  # shape (N,)
    
    q_mat = torch.clamp(torch.round(mat / scales.unsqueeze(1)), qmin, qmax).to(torch.int8)  # shape (N, M)
    
    scales = scales.to(torch.float32)
    return q_mat, scales

def de_quantize_row_matrix_int8_symmetric(q_mat:torch.Tensor, scale:torch.Tensor, out_dtype=torch.float16):
    """
    Dequantize int8 matrix to float on a per-row basis.
    q_mat: input int8 tensor (shape (N, M))
    scales: scale factors for each row (shape (N,))
    """
    output = q_mat.to(torch.float32) 
    output = output * scale[:, None]
    output = output.to(out_dtype)
    return output

def quantized_column_matrix_int_symmetric(mat:torch.Tensor):
    """
    Symmetric quantization to int8 on a per-column basis.
    mat: input float tensor (e.g., torch.float32 or torch.bfloat16)
    """
    N, M = mat.shape
    qmin = -128
    qmax = 127
    
    max_vals, _ = torch.max(torch.abs(mat), dim=0, keepdim=True)  # shape (1, M)
    scales = (max_vals / qmax).squeeze(0)  # shape (M,)
    
    q_mat = torch.clamp(torch.round(mat / scales.unsqueeze(0)), qmin, qmax).to(torch.int8)  # shape (N, M)
    
    scales = scales.clone().detach().to(torch.float32)
    return q_mat, scales


def de_quantized_column_matrix_int8_symmetric(q_mat:torch.Tensor, scale:torch.Tensor, out_dtype=torch.float16):
    """
    Dequantize int8 matrix to float on a per-column basis.
    q_mat: input int8 tensor (shape (N, M))
    scales: scale factors for each column (shape (M,))
    """
    output = q_mat.to(torch.float32) 
    output = output * scale[None, :]
    output = output.to(out_dtype)
    return output

def de_quantize_row_matrix_int8_symmetric_matmul(q_mat:torch.Tensor, x_scale:torch.Tensor, w_scale: torch.Tensor,\
                                        out_dtype=torch.float16):
    """
    Dequantize int8 matrix to float on a per-row basis.
    q_mat: input int8 tensor (shape (N, M))
    scales: scale factors for each row (shape (N,))
    """
    output = q_mat.to(torch.float32) 
    output = output * x_scale[:, None] * w_scale[None, :]
    output = output.to(out_dtype)
    return output

In [34]:
N = 1024

W = torch.randn(N, N, device='cuda', dtype=d_type)
X = torch.randn(N, N, device='cuda', dtype=d_type)

A = torch.matmul(X, W)
print(f"Shape of A: {A.shape}, dtype: {A.dtype}")

Shape of A: torch.Size([1024, 1024]), dtype: torch.float16


In [35]:
X_q, x_scale = quantize_row_matrix_int8_symmetric(X)
print(f"Shape of X_q: {X_q.shape}, dtype: {X_q.dtype}")
print(f"Shape of x_scale: {x_scale.shape}, dtype: {x_scale.dtype}")

W_q, w_scale = quantized_column_matrix_int_symmetric(W)
print(f"Shape of W_q: {W_q.shape}, dtype: {W_q.dtype}")
print(f"Shape of w_scale: {w_scale.shape}, dtype: {w_scale.dtype}")

Shape of X_q: torch.Size([1024, 1024]), dtype: torch.int8
Shape of x_scale: torch.Size([1024]), dtype: torch.float32
Shape of W_q: torch.Size([1024, 1024]), dtype: torch.int8
Shape of w_scale: torch.Size([1024]), dtype: torch.float32


In [21]:
# A_q = bnb.functional.int8_linear_matmul(X_q, W_q)
A_q = torch.matmul(X_q.float(), W_q.float())  # TODO
print(f"Shape of A_q: {A_q.shape}, dtype: {A_q.dtype}")

Shape of A_q: torch.Size([1024, 1024]), dtype: torch.float32


In [22]:
A_deq = de_quantize_row_matrix_int8_symmetric_matmul(A_q, x_scale, w_scale)
print(f"Shape of A_deq: {A_deq.shape}, dtype: {A_deq.dtype}")

Shape of A_deq: torch.Size([1024, 1024]), dtype: torch.float16


In [23]:
if torch.allclose(A, A_deq, rtol=2.0, atol=2.0):
    print("Correct !! \n")
else:
    print("WRONG - Dequantized matrix is NOT close !! \n")
    
error_l2 = quantization_error_l2_norm(A, A_deq)
print(f"Quantization L2 norm error (hierarchical quantization int8): {error_l2.item():.6f}")

error_mse = quantization_error_mse(A, A_deq)
print(f"Quantization MSE (hierarchical quantization int8): {error_mse.item():.6f}")

error_kl = quantization_error_kl_divergence(A, A_deq)
print(f"Quantization KL divergence (hierarchical quantization int8): {error_kl.item():.6f}")

Correct !! 

Quantization L2 norm error (hierarchical quantization int8): 365.500000
Quantization MSE (hierarchical quantization int8): 0.127319
Quantization KL divergence (hierarchical quantization int8): 0.076195


# 5. Matmul - Matrix quantization

In [24]:
N = 1024
d_type = torch.float16

W = torch.randn(N, N, device='cuda', dtype=d_type)
X = torch.randn(N, N, device='cuda', dtype=d_type)

A = torch.matmul(X, W)
print(f"Shape of A: {A.shape}, dtype: {A.dtype}")
print(f"Min and Max of A: {torch.min(A).item():.6f}, {torch.max(A).item():.6f}")

Shape of A: torch.Size([1024, 1024]), dtype: torch.float16
Min and Max of A: -155.125000, 148.625000


In [25]:
X_q, x_scale = quantize_matrix_symmetric_int8(X)
W_q, w_scale = quantize_matrix_symmetric_int8(W)

print(f"Shape of X_q: {X_q.shape}, dtype: {X_q.dtype}")
print(f"Shape of x_scale: {x_scale.shape}, dtype: {x_scale.dtype}")
print(f"Shape of W_q: {W_q.shape}, dtype: {W_q.dtype}")
print(f"Shape of w_scale: {w_scale.shape}, dtype: {w_scale.dtype}")

Shape of X_q: torch.Size([1024, 1024]), dtype: torch.int8
Shape of x_scale: torch.Size([]), dtype: torch.float32
Shape of W_q: torch.Size([1024, 1024]), dtype: torch.int8
Shape of w_scale: torch.Size([]), dtype: torch.float32


/tmp/ipykernel_326260/2675896001.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  scale = torch.tensor(scale, dtype=torch.float32)


In [26]:
A_deq = torch.matmul(X_q.float(), W_q.float())  * x_scale * w_scale  # TODO 
A_deq = A_deq.to(d_type)
print(f"Shape of A_deq: {A_deq.shape}, dtype: {A_deq.dtype}")

Shape of A_deq: torch.Size([1024, 1024]), dtype: torch.float16


In [27]:
if torch.allclose(A, A_deq, rtol=2.0, atol=2.0):
    print("Correct !! \n")
else:
    print("WRONG - Dequantized matrix is NOT close !! \n")
    
error_l2 = quantization_error_l2_norm(A, A_deq)
print(f"Quantization L2 norm error (matrix symmetric int8): {error_l2.item():.6f}")

error_mse = quantization_error_mse(A, A_deq)
print(f"Quantization MSE (matrix symmetric int8): {error_mse.item():.6f}")

error_kl = quantization_error_kl_divergence(A, A_deq)
print(f"Quantization KL divergence (matrix symmetric int8): {error_kl.item():.6f}")

Correct !! 

Quantization L2 norm error (matrix symmetric int8): 512.500000
Quantization MSE (matrix symmetric int8): 0.250488
Quantization KL divergence (matrix symmetric int8): 4.947989


# 6. Matmul - Tensor quantization

In [28]:
H = 32
N = 512

W = torch.randn((H, N, N), device='cuda', dtype=d_type)
X = torch.randn((H, N, N), device='cuda', dtype=d_type)

A = torch.bmm(X, W)
print(f"Shape of A: {A.shape}, dtype: {A.dtype}")

Shape of A: torch.Size([32, 512, 512]), dtype: torch.float16


In [29]:
X_q, x_scale = quantize_tensor_symmetric_int8(X)
W_q, w_scale = quantize_tensor_symmetric_int8(W)

print(f"Shape of X_q: {X_q.shape}, dtype: {X_q.dtype}")
print(f"Shape of x_scale: {x_scale.shape}, dtype: {x_scale.dtype}")
print(f"Shape of W_q: {W_q.shape}, dtype: {W_q.dtype}")
print(f"Shape of w_scale: {w_scale.shape}, dtype: {w_scale.dtype}")

Shape of X_q: torch.Size([32, 512, 512]), dtype: torch.int8
Shape of x_scale: torch.Size([]), dtype: torch.float32
Shape of W_q: torch.Size([32, 512, 512]), dtype: torch.int8
Shape of w_scale: torch.Size([]), dtype: torch.float32


/tmp/ipykernel_326260/315809518.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  scale = torch.tensor(scale, dtype=torch.float32)


In [30]:
A_deq = torch.bmm(X_q.float(), W_q.float())  * x_scale * w_scale
A_deq = A_deq.to(d_type)
print(f"Shape of A_deq: {A_deq.shape}, dtype: {A_deq.dtype}")

Shape of A_deq: torch.Size([32, 512, 512]), dtype: torch.float16


In [31]:
if torch.allclose(A, A_deq, rtol=2.0, atol=2.0):
    print("Correct !! \n")
else:
    print("WRONG - Dequantized matrix is NOT close !! \n")
    
error_l2 = quantization_error_l2_norm(A, A_deq)
print(f"Quantization L2 norm error (tensor symmetric int8): {error_l2.item():.6f}")

error_mse = quantization_error_mse(A, A_deq)
print(f"Quantization MSE (tensor symmetric int8): {error_mse.item():.6f}")

error_kl = quantization_error_kl_divergence(A, A_deq)
print(f"Quantization KL divergence (tensor symmetric int8): {error_kl.item():.6f}")

Correct !! 

Quantization L2 norm error (tensor symmetric int8): 1161.000000
Quantization MSE (tensor symmetric int8): 0.160645
Quantization KL divergence (tensor symmetric int8): 7.059552
